# Preparacion reproducible en Colab
Seleccionar runtime GPU T4 para entrenamiento/inferencia. Repositorio privado: cargar un ZIP autorizado mediante selector; no escribir tokens en celdas. Se trabaja desde raiz del proyecto. Las salidas vacias no constituyen evidencia de ejecucion.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(archive) as z:
    z.extractall('/content/proyecto')
roots = list(Path('/content/proyecto').rglob('requirements.txt'))
assert len(roots) == 1, 'El ZIP debe contener una sola copia del proyecto'
os.chdir(roots[0].parent)
print(Path.cwd())

In [ ]:
%pip install torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124
%pip install -r requirements.txt
!python -m pip check

In [ ]:
import torch, transformers, peft, accelerate, bitsandbytes, datasets
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
print('transformers', transformers.__version__, 'PEFT', peft.__version__)
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

# Evaluacion real
Ejecutar primero baseline, antes de entrenamiento. Configurar MySQL 8.4.10 de solo lectura y cargar esquemas/fixtures de validacion. Las preguntas necesitan revision humana y nunca deben formar parte de entrenamiento. Ejecutar la misma CLI y prompt para base, A y B, y despues estrategia directa/correccion. Ver README para comandos definitivos.

In [ ]:
import getpass
os.environ['MYSQL_HOST'] = input('Host MySQL: ')
os.environ['MYSQL_PORT'] = input('Puerto [3306]: ') or '3306'
os.environ['MYSQL_USER'] = input('Usuario de solo lectura: ')
os.environ['MYSQL_PASSWORD'] = getpass.getpass('Contrasena: ')

In [ ]:
import json
from pathlib import Path
os.environ['MYSQL_READONLY_PASSWORD'] = getpass.getpass('Contrasena readonly del proyecto: ')
for name, adapter in [('baseline', None), ('lora_a', 'adapters_experimentos/config_a'), ('lora_b', 'adapters_experimentos/config_b')]:
    cfg = {'modelo_base':'Qwen/Qwen2.5-Coder-1.5B-Instruct','adapter': adapter or 'adapter/', 'usar_adapter':adapter is not None, 'semilla':2026}
    Path(f'config_{name}.json').write_text(json.dumps(cfg,indent=2))
!python scripts/evaluate.py --dataset datos/validacion/aceptados.jsonl --config config_baseline.json --strategy ambas


Tras baseline, ejecutar A y B usando las rutas adapters_experimentos/config_a y config_b. No interpretar una celda sin salida como experimento realizado. Guardar CSV por pregunta y resumenes reales. Promover pesos solamente despues de comparar EX y latencias.

Ejecutar las siguientes celdas **despues** del notebook 02, en runtime con los pesos reales.

In [ ]:
!python scripts/evaluate.py --dataset datos/validacion/aceptados.jsonl --config config_baseline.json --config config_lora_a.json --config config_lora_b.json --strategy ambas


In [ ]:
import shutil
shutil.make_archive('/content/evaluacion', 'zip', '.', 'resultados')
files.download('/content/evaluacion.zip')